# Input Optimization Performance Eval

In [ ]:
import pathlib as pl
import pywatershed

repo_root = pywatershed.constants.__pywatershed_root__.parent

In [ ]:
# Instantiate control, params, inputs, model and run it to completion with budget choice
def proc_model_performance(process, domain, calc_method, imbalance_behavior: str = None, load_n_time_batches: int = 1):
    
    domain_dir = repo_root / f"test_data/{domain}"
    input_dir = domain_dir / "output"
    
    params = pywatershed.PrmsParameters.load(domain_dir / "myparam.param")
    control = pywatershed.Control.load(domain_dir / "control.test", params=params)

    input_variables = {}
    for key in process.get_inputs():
      nc_path = input_dir / f"{key}.nc"
      input_variables[key] = nc_path

    proc_model = process(
      control,
      **input_variables,
      imbalance_behavior=imbalance_behavior,
      calc_method=calc_method,
      load_n_time_batches=load_n_time_batches,
    )

    for istep in range(control.n_times):
      control.advance()
      proc_model.advance()
      proc_model.calculate(float(istep))

    proc_model.finalize()

    return

# Generate profiling data

In [ ]:
domains = ['drb_2yr', 'ucb_2yr', 'hru_1']
calc_methods = ['numba', 'fortran', 'numpy']
batch = {'one': 1, 'none': None}
processes = [pywatershed.PRMSCanopy, pywatershed.PRMSChannel, pywatershed.PRMSGroundwater,]
results = []
ii = 0 
for pp in processes:
    for dd in domains:
        for cc in calc_methods:
            for bb_key, bb_val in batch.items():
                print(ii)
                ii += 1
                if (pp.__name__ != "PRMSGroundwater") and (cc == 'jax'):
                    continue  # only implemented for PRMSGroundwater so far
                print('\n', pp.__name__, dd, cc, bb_key)
                result = %timeit -o proc_model_performance(pp, dd, cc, load_n_time_batches=bb_val)
                results += [{(pp.__name__, dd, cc, bb_key): result}]

In [ ]:
remap_io = {'one': 'IO optimzed', 'none': 'IO default'}
results_post = {}
for rr in results:
    kk = list(list(rr.keys())[0])
    kk[3] = remap_io[kk[3]]
    vv = list(rr.values())[0]
    results_post[tuple(kk)] = {'mean': vv.average, 'stdev': vv.stdev, 'N': vv.repeat}

In [ ]:
results_post

In [ ]:
import pandas as pd
pd.options.plotting.backend = 'holoviews'

In [ ]:
results_df = pd.DataFrame(results_post).T
results_df.index.set_names(names = ["process", "domain", "calc", "io"], inplace=True)

In [ ]:
results_df

In [ ]:
for pp in ["PRMSGroundwater", 'PRMSCanopy', "PRMSChannel"]:
    for dd in domains:
        proc_df = results_df.loc[pp, dd, slice(None), slice(None), slice(None)] 
        display(
            proc_df.plot.bar(
                #by='domain', subplots=True
                rot=40,
            ).opts(
                title=f"{pp}: {dd}", height=450, width=600, 
                ylabel='Mean Time (seconds)', 
                xlabel='Calculation Method: IO Method',
                fontscale=1.5))

In [ ]:
# (proc_df.plot.bar().opts(title=pp) * 
# proc_df.hvplot.errorbars(y='mean', yerr1='stdev'))

In [ ]:
# proc_df_ri = proc_df.reset_index()
# proc_df_ri

In [ ]:
# proc_df_ri.plot.bar(y='mean') * proc_df_ri.hvplot.errorbars(x='index',y='mean', yerr1='stdev')

# Performance profiling


In [ ]:
%load_ext snakeviz

In [ ]:
%%snakeviz
proc_model_performance(pywatershed.PRMSGroundwater, 'hru_1', 'numpy', load_n_time_batches = None)

In [ ]:
%%snakeviz
proc_model_performance(pywatershed.PRMSGroundwater, 'hru_1', 'numpy', load_n_time_batches = 1)

## Notes on profiles

The IO is on a per time basis, so IO dominates the hru_1 domain. Dramatic reductions in time result from reading all data at the initial time. 